# Example data preparation pipeline on tahoe.

## Prepare Notebook.

In [2]:
%reload_ext autoreload
%autoreload 2

### Import Libraries.

In [3]:
from sc_flow.data import DataManager
import anndata as ad

### Define constants.

In [4]:
H5AD_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/tahoe_analysis/subsample/tahoe_sorted.h5ad"

## Data.

### Read data from disk.

In [5]:
adata = ad.read_h5ad(H5AD_PATH)
adata

AnnData object with n_obs × n_vars = 89423257 × 3
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'id', 'drugname_drugconc', 'drug', 'INT_ID', 'NUM.SNPS', 'NUM.READS', 'demuxlet_call', 'BEST.GUESS', 'BEST.LLK', 'NEXT.GUESS', 'NEXT.LLK', 'DIFF.LLK.BEST.NEXT', 'BEST.POSTERIOR', 'SNG.POSTERIOR', 'cell_line', 'SNG.BEST.LLK', 'SNG.NEXT.GUESS', 'SNG.NEXT.LLK', 'SNG.ONLY.POSTERIOR', 'DBL.BEST.GUESS', 'DBL.BEST.LLK', 'DIFF.LLK.SNG.DBL', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'cell_line_orig', 'pass_filter', 'cell_name', 'dosage', 'plate'

### Add dummy representation to adata for drugs.

In [6]:
adata.uns["drug_rep"] = {}

In [15]:
adata.obs['drug'] = adata.obs['drug'].astype('category')
adata.obs['cell_line'] = adata.obs['cell_line'].astype('category')

### Initialize data manager.

In [17]:
dm = DataManager(
    conditions={"drug": ["drug"]},
    conditions_reps={"drug": "drug_rep"},
    groups=["cell_line"],
    groups_encoding={"cell_line": "one-hot"},
)
train_collection = dm.compile_adata(adata)


Getting matched distributions...


KeyboardInterrupt: 

In [ ]:
df = dm.get_distribution_data(adata).ann_df

In [20]:
df

,drug,cell_line
72_005_017-lib_841,4EGI-1,CVCL_0023
72_036_191-lib_841,4EGI-1,CVCL_0023
72_045_112-lib_841,4EGI-1,CVCL_0023
72_058_077-lib_841,4EGI-1,CVCL_0023
72_075_175-lib_841,4EGI-1,CVCL_0023
...,...,...
72_147_183-lib_2497,γ-Oryzanol,CVCL_C466
72_158_010-lib_2497,γ-Oryzanol,CVCL_C466
72_167_022-lib_2497,γ-Oryzanol,CVCL_C466
72_183_038-lib_2497,γ-Oryzanol,CVCL_C466


In [26]:
import time
import pandas as pd
import numpy as np

In [ ]:
import time
import numpy as np
import pandas as pd

df = adata.obs[['drug', 'cell_line']].copy()
df['drug'] = df['drug'].astype('category')
df['cell_line'] = df['cell_line'].astype('category')

# Simulate: groups=['cell_line'], conditions=['cell_line', 'drug']
# i.e. conditions has 2 columns

registry = {
    'groups': ['cell_line'],
    'conditions': ['cell_line', 'drug'],
}

t0 = time.time()

all_levels = [df.index]
all_codes = [np.arange(len(df))]
all_names = ['base']

for level_name, level_cols in registry.items():
    for col in sorted(level_cols):
        cat = df[col].cat
        all_levels.append(cat.categories)
        all_codes.append(cat.codes.values)
        all_names.append((level_name, col))

mi = pd.MultiIndex(levels=all_levels, codes=all_codes, names=all_names)
print(f"MultiIndex() direct: {time.time() - t0:.3f}s, n_levels={mi.nlevels}, len={len(mi)}", flush=True)
print(f"Names: {mi.names}", flush=True)

# Verify structure
for name in mi.names:
    vals = mi.get_level_values(name)
    print(f"  {name}: {vals.nunique()} unique values", flush=True)

In [24]:
arrays = [df.index, df['cell_line'], df['drug']]
names = ['base', ('groups', 'cell_line'), ('conditions', 'drug')]
mi = pd.MultiIndex.from_arrays(arrays, names=names)


KeyboardInterrupt: 